# Predicting Airbnb Occupancy and Pricing Using Listing and Neighborhood Demographic Data

# Machine Learning Project Notebook
## Authors: Roman Shrestha and Tanish Pradhan Wong Ah Sui

In [ ]:
import pandas as pd
import requests
import time
import os
import json
import re
from pathlib import Path
import warnings
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.options.display.float_format = '{:.2f}'.format

In [ ]:
CENSUS_API_KEY = os.environ.get("CENSUS_API_KEY", "")
ACS_YEAR = 2023

In [ ]:
def load_listings(data_dir):
    dfs = []
    for path in Path(data_dir).glob("*.csv"):
        df = pd.read_csv(path)
        df["city"] = path.stem  # e.g. albany_NY
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

airbnb = load_listings("data/city_lists/")

In [ ]:
# geocode lat/lon to census tracts
def geocode_to_tract_batch(df,batch_size=100):
    state_fips_list, county_fips_list, tract_fips_list, full_fips_list = [], [], [], []

    for i, row in df.iterrows():
        lat, lon = row["latitude"], row["longitude"]

        if pd.isna(lat) or pd.isna(lon):
            state_fips_list.append(None)
            county_fips_list.append(None)
            tract_fips_list.append(None)
            full_fips_list.append(None)
            continue

        params = {
            "x": lon,"y": lat,
            "benchmark": "Public_AR_Current",
            "vintage": "Census2020_Current",
            "format": "json"
        }

        try:
            resp = requests.get("https://geocoding.geo.census.gov/geocoder/geographies/coordinates",params=params, timeout=30)
            tracts = resp.json()["result"]["geographies"].get("Census Tracts", [])

            if tracts:
                t = tracts[0]
                state_fips_list.append(t["STATE"])
                county_fips_list.append(t["COUNTY"])
                tract_fips_list.append(t["TRACT"])
                full_fips_list.append(t["STATE"] + t["COUNTY"] + t["TRACT"])
            else:
                state_fips_list.extend([None, None, None, None])
        except:
            state_fips_list.extend([None, None, None, None])

    df["state_fips"] = state_fips_list
    df["county_fips"] = county_fips_list
    df["tract_fips"] = tract_fips_list
    df["full_tract_fips"]= full_fips_list
    return df

In [ ]:
# load listings and geocode
airbnb = load_listings("data/listings/")
airbnb = geocode_to_tract_batch(airbnb)

airbnb.to_csv("data/listings_geocoded.csv", index=False)

In [ ]:
# ACS 5-year variables that are relevant
# https://api.census.gov/data/2023/acs/acs5/variables.html
ACS_VARIABLES = {
    #income and poverty
    "B19013_001E": "median_household_income",
    "B17001_002E": "population_below_poverty",
    "B17001_001E": "population_poverty_universe",
    #housing
    "B25064_001E": "median_gross_rent",
    "B25003_001E": "total_housing_tenure",
    "B25003_003E": "renter_occupied_units",
    "B25077_001E": "median_home_value",
    #demographics
    "B01003_001E": "total_population",
    "B01002_001E": "median_age",
    #race/ethnicity
    "B03002_003E": "white_alone_not_hispanic",
    "B03002_004E": "black_alone",
    "B03002_006E": "asian_alone",
    "B03002_012E": "hispanic_latino",
    "B03002_001E": "total_race_universe",
    #education (25+)
    "B15003_022E": "bachelors_degree",
    "B15003_023E": "masters_degree",
    "B15003_024E": "professional_degree",
    "B15003_025E": "doctorate_degree",
    "B15003_001E": "total_education_universe",
    #employment
    "B23025_003E": "civilian_labor_force",
    "B23025_005E": "unemployed",
}

In [ ]:
# fetch ACS data
def fetch_acs_data(df, api_key, year=2023):
    tracts_df = df[["state_fips", "county_fips", "tract_fips"]].drop_duplicates().dropna()
    var_string = ",".join(ACS_VARIABLES.keys())

    all_tract_data = []
    for (state, county), group in tracts_df.groupby(["state_fips", "county_fips"]):
        tract_list = group["tract_fips"].tolist()
        try:
            resp = requests.get(
                f"https://api.census.gov/data/{year}/acs/acs5",
                params={"get": f"NAME,{var_string}", "for": "tract:*",
                        "in":f"state:{state} county:{county}", "key": api_key},
                timeout=30
            )
            data = resp.json()
            county_df = pd.DataFrame(data[1:], columns=data[0])
            county_df=county_df[county_df["tract"].isin(tract_list)]
            county_df["full_tract_fips"] = county_df["state"] + county_df["county"] + county_df["tract"]
            all_tract_data.append(county_df)
        except Exception as e:
            print(f"error")
    
    census_df = pd.concat(all_tract_data, ignore_index=True)
    census_df.rename(columns=ACS_VARIABLES, inplace=True)

    for col in ACS_VARIABLES.values():
        census_df[col] =pd.to_numeric(census_df[col], errors="coerce")

    census_df["poverty_rate"] =census_df["population_below_poverty"] / census_df["population_poverty_universe"]
    census_df["pct_renter_occupied"] = census_df["renter_occupied_units"] / census_df["total_housing_tenure"]
    census_df["pct_white"]= census_df["white_alone_not_hispanic"] / census_df["total_race_universe"]
    census_df["pct_black"]= census_df["black_alone"] / census_df["total_race_universe"]
    census_df["pct_asian"] = census_df["asian_alone"] / census_df["total_race_universe"]
    census_df["pct_hispanic"] = census_df["hispanic_latino"] / census_df["total_race_universe"]
    census_df["pct_college_educated"] = (
        census_df[["bachelors_degree", "masters_degree", "professional_degree", "doctorate_degree"]].sum(axis=1)
        / census_df["total_education_universe"]
    )
    census_df["unemployment_rate"] =census_df["unemployed"] / census_df["civilian_labor_force"]

    keep = ["full_tract_fips", "median_household_income", "median_gross_rent", "median_home_value",
            "total_population", "median_age", "poverty_rate", "pct_renter_occupied",
            "pct_white", "pct_black", "pct_asian", "pct_hispanic",
            "pct_college_educated", "unemployment_rate"]

    return census_df[keep]

In [ ]:
# fetch census data
census = fetch_acs_data(airbnb,CENSUS_API_KEY, year=ACS_YEAR)
census.to_csv("data/census_tract_data.csv", index=False)

In [ ]:
# merge airbnb + census
def merge_data(airbnb_df, census_df):
    return airbnb_df.merge(census_df, on="full_tract_fips", how="left")

In [ ]:
# merge
merged = merge_data(airbnb,census)
merged.to_csv("data/airbnb_with_census.csv", index=False)

In [ ]:
airbnb_with_census = merged.copy()

In [ ]:
cols_to_keep = [
    "price",
    "estimated_occupancy_l365d",
    "estimated_revenue_l365d",
    "review_scores_rating",
    "reviews_per_month",
    "property_type",
    "room_type",
    "accommodates",
    "bedrooms",
    "beds",
    "bathrooms_text",
    "amenities",
    "minimum_nights",
    "maximum_nights",
    "instant_bookable",
    "availability_365",
    "availability_30",
    "availability_60",
    "availability_90",
    "minimum_minimum_nights",
    "host_id",
    "host_is_superhost",
    "host_response_time",
    "host_response_rate",
    "host_acceptance_rate",
    "host_identity_verified",
    "host_has_profile_pic",
    "calculated_host_listings_count",
    "hosts_time_as_host_years",
    "number_of_reviews",
    "review_scores_cleanliness",
    "review_scores_location",
    "review_scores_value",
    "review_scores_communication",
    "city",
    "latitude",
    "longitude",
    "neighbourhood_cleansed",
    "full_tract_fips",
    "state_fips",
    "county_fips",
    "tract_fips",
    "median_household_income",
    "median_gross_rent",
    "median_home_value",
    "total_population",
    "median_age",
    "poverty_rate",
    "pct_renter_occupied",
    "pct_white",
    "pct_black",
    "pct_asian",
    "pct_hispanic",
    "pct_college_educated",
    "unemployment_rate",
    "listing_url",
    "name",
    "description"
]

airbnb_with_census = airbnb_with_census[cols_to_keep]


In [ ]:
# missing values
missing = pd.DataFrame({
    "missing_n": airbnb_with_census[cols_to_keep].isnull().sum(),
    "missing_pct": (airbnb_with_census[cols_to_keep].isnull().sum() / len(airbnb_with_census) * 100).round(2)
})
missing[missing["missing_n"] >0].sort_values("missing_pct", ascending=False)

In [ ]:
def clean_listings(df):

    # bools
    bool_cols = ["instant_bookable","host_is_superhost", "host_identity_verified", "host_has_profile_pic"]
    for col in bool_cols:
        df[col] = df[col].map({"t": True, "f": False})

    sentinel_cols = ["median_household_income", "median_gross_rent", 
                 "median_home_value", "estimated_revenue_l365d"]
    for col in sentinel_cols:
        df[col] = df[col].where(df[col] >= 0, other=pd.NA)

    if df["price"].dtype == object:
        df["price"] = df["price"].str.replace(r"[$,]", "", regex=True).astype(float)
    p95 = df["price"].quantile(0.95)
    df["price"] = df["price"].clip(upper=p95)

    p95_rev = df["estimated_revenue_l365d"].quantile(0.95)
    df["estimated_revenue_l365d"]= df["estimated_revenue_l365d"].clip(upper=p95_rev)

    # get only the number
    df["bathrooms"] = df["bathrooms_text"].str.extract(r"([\d.]+)").astype(float)
    df.drop(columns="bathrooms_text", inplace=True)

    # amenities
    df = df[df["amenities"].apply(lambda x: isinstance(x, str) or isinstance(x, list))] # removing some amenities that are int

    def parse_amenities(val):
        try:
            result = json.loads(val) if isinstance(val, str) else list(val)
            return [str(a) for a in result] 
        except:
            return []

    amenities_parsed = df["amenities"].apply(parse_amenities)

    for amenity, col in [("Wifi", "has_wifi"), ("Free parking", "has_parking"),
                          ("Pool", "has_pool"), ("Gym", "has_gym")]:
        df[col] = amenities_parsed.apply(lambda x: any(amenity.lower() in a.lower() for a in x))
    df.drop(columns="amenities", inplace=True)

    # response time 
    response_order = {"within an hour": 1, "within a few hours": 2,
                      "within a day": 3, "a few days or more": 4}
    df["host_response_time"] = df["host_response_time"].map(response_order)

    # missing values
    # impute median by room type
    df["beds"] = df.groupby("room_type")["beds"].transform(lambda x: x.fillna(x.median()))

    df["bedrooms"]= df.groupby("property_type")["bedrooms"].transform(lambda x: x.fillna(x.median()))

    # host response if unknown category then 0
    df["host_response_time"] = df["host_response_time"].fillna(0) 
    df["host_response_rate"] = df["host_response_rate"].fillna(0)
    df["host_acceptance_rate"] = df["host_acceptance_rate"].fillna(0)
    df["host_is_superhost"] = df["host_is_superhost"].fillna(False)

    df["maximum_nights"]= df["maximum_nights"].clip(upper=365)

    # droping rows where census tracks did not match
    df.dropna(subset=["poverty_rate"], inplace=True)

    return df

airbnb_clean = clean_listings(airbnb_with_census.copy())

In [ ]:
airbnb_clean.to_csv("data/airbnb_cleaned.csv")

In [ ]:
sample_airbnb = airbnb_clean.sample(1000)
sample_airbnb.to_csv("data/sample_airbnb.csv")

In [ ]:
# basic info
print(f"Observations: {airbnb_clean.shape[0]:,}")
print(f"Variables: {airbnb_clean.shape[1]}")

# summary of numerical variables
airbnb_clean.describe().T[["mean", "std", "min", "50%", "max"]].round(2)

In [ ]:
# categorical variables
cat_cols = airbnb_clean.select_dtypes(include=["object", "bool"]).columns
for col in cat_cols:
    print(f"\n{col} ({airbnb_clean[col].nunique()} unique):")
    print(airbnb_clean[col].value_counts(normalize=True).mul(100).round(1).head(5).to_string())

In [ ]:
# distributions of key numeric variables
plot_cols = ["price", "estimated_revenue_l365d", "estimated_occupancy_l365d",
             "review_scores_rating", "median_household_income", "poverty_rate"]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.flatten(), plot_cols):
    airbnb_clean[col].dropna().hist(bins=50, ax=ax)
    ax.set_title(col)
    ax.set_xlabel("")
plt.tight_layout()
plt.show()

In [ ]:
# listings by city
airbnb_clean["city"].value_counts().to_frame("count").assign(
    pct=lambda x: (x["count"] / x["count"].sum() * 100).round(1)
)